In [ ]:
import cv2
import numpy as np
import pandas as pd
from pathlib import Path
import lib
import os
from matplotlib import pyplot as plt

In [ ]:
def recuperarPares(csv_path):
    print(csv_path)
    df = pd.read_csv(csv_path, sep=';', engine="python")
    object_single = df[["X", "Y", "Z"]].to_numpy(dtype=np.float32)

    obj_points = []
    img_points = []

    for i in range(4):
        image_single = df[[f"u_{i}", f"v_{i}"]].to_numpy(dtype=np.float32)

        obj_points.append(object_single.copy())
        img_points.append(image_single.copy())

    return obj_points, img_points

In [ ]:
data_path = './salguero'

img_bgr = cv2.imread(str(os.path.join(data_path, 'img_10_22_40_781.jpg')))
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

In [ ]:
obj_points, img_points = recuperarPares(os.path.join(data_path, 'world_img_pts.csv'))

In [ ]:
worldPts = obj_points[1].T[:2,:]
videoPts = img_points[1].T

In [ ]:
# la homografia de ejemplo ya viene dada
H = np.array([[ 1.09443793e+02,  4.28987478e+01,  5.92685821e+02],
 [ 2.45115762e+00,  2.91349288e+00,  5.41092448e+02],
 [-1.02639841e-02,  2.98016564e-02,  1.00000000e+00]])

In [ ]:
# paso las coordenadas del mundo real a coordenadas homogeneas (le agrego la fila de ceros)
h_world = np.vstack([worldPts, np.ones(worldPts.shape[1])])

In [ ]:
# proyecto las coordenadas a coordenadas homogeneas del plano imagen
h_img = H @ h_world

In [ ]:
# le aplico el alpha para corregir la posición u,v
nlin, ncol = h_img.shape
proj_img = h_img[:3, :] / h_img[2:3, :]

In [ ]:
figsize=(12, 8)

U, V = videoPts[0,:], videoPts[1,:]
X, Y = proj_img[0,:], proj_img[1,:]

marker_size=35
fig, ax = plt.subplots(figsize=figsize)

ax.imshow(img_rgb)
ax.scatter(U, V, s=marker_size, marker="o", color='blue')
ax.scatter(X, Y, s=marker_size, marker="x", color='red')

ax.set_title("Fotografía con puntos originales (circulos azules) y proyectados del mundo real (cruces rojas)")
ax.set_xlabel("x / columna de imagen")
ax.set_ylabel("y / fila de imagen")
ax.grid(False)
plt.tight_layout()
